# 📖 Module 06: Retrievers

## GenAI L2 Exam Preparation

**Topics Covered:**
- Basic retriever from vector store
- Similarity search vs MMR
- Dense vs Sparse retrieval (BM25)
- Multi-query retriever
- Contextual compression retriever
- Ensemble retriever (hybrid search)
- Metadata filtering

**Source Material:** Class 35 (Chunking & Retriever), Class 36 (Retriever), Class 37 (Advanced Retriever)

---

## 1. What is a Retriever?

A retriever is the component that **finds relevant documents** for a given query from a knowledge base.

```
User Query → Retriever → Top-K Relevant Documents → LLM → Answer
```

### Why Retrieval Quality Matters
```
Good retrieval + Good LLM = Great answers ✅
Bad retrieval + Great LLM = Bad answers ❌  ← The LLM can't fix bad context!
```

### 🎯 Exam Tip
> Retrieval is the **most impactful component** in a RAG pipeline.  
> A better retriever improves results more than a better LLM.

In [ ]:
# Setup: Create a vector store for retriever demos
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It enhances LLM responses with external knowledge.", metadata={"topic": "RAG", "section": "intro"}),
    Document(page_content="The RAG pipeline has two phases: indexing (offline) and querying (online).", metadata={"topic": "RAG", "section": "architecture"}),
    Document(page_content="Vector databases like FAISS and ChromaDB store embeddings for similarity search.", metadata={"topic": "VectorDB", "section": "overview"}),
    Document(page_content="FAISS is developed by Facebook AI Research and is known for its speed.", metadata={"topic": "VectorDB", "section": "FAISS"}),
    Document(page_content="Cosine similarity measures the angle between two vectors, ranging from -1 to 1.", metadata={"topic": "Embeddings", "section": "metrics"}),
    Document(page_content="RecursiveCharacterTextSplitter is the recommended default chunking strategy.", metadata={"topic": "Chunking", "section": "methods"}),
    Document(page_content="Fine-tuning changes model weights, while RAG retrieves external information at inference.", metadata={"topic": "RAG", "section": "comparison"}),
    Document(page_content="BM25 is a sparse retrieval method based on term frequency that excels at keyword matching.", metadata={"topic": "Retrieval", "section": "sparse"}),
    Document(page_content="MMR (Maximal Marginal Relevance) balances relevance with diversity in retrieved results.", metadata={"topic": "Retrieval", "section": "diversity"}),
    Document(page_content="Hybrid search combines dense semantic search with sparse keyword search for best results.", metadata={"topic": "Retrieval", "section": "hybrid"}),
]

vectorstore = FAISS.from_documents(docs, embeddings)
print(f"✅ Created vector store with {len(docs)} documents")

## 2. Basic Retriever (Similarity Search)

In [ ]:
# Create a basic retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Default
    search_kwargs={"k": 3}     # Return top 3
)

# Use the retriever
query = "What is the difference between RAG and fine-tuning?"
results = retriever.invoke(query)

print(f"🔍 Query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    Topic: {doc.metadata['topic']} | Section: {doc.metadata['section']}\n")

## 3. ⭐ MMR (Maximal Marginal Relevance) — Diversity!

**Problem**: Similarity search can return very similar (redundant) documents.  
**Solution**: MMR balances **relevance** with **diversity**.

### How MMR Works
```
MMR Score = λ × Relevance(doc, query) - (1-λ) × max Similarity(doc, already_selected)

λ (lambda) close to 1 → prioritize relevance (like regular similarity search)
λ (lambda) close to 0 → prioritize diversity (maximize difference from already selected docs)
```

In [ ]:
# MMR Retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,             # Return 3 documents
        "fetch_k": 10,      # Fetch 10 candidates first, then select 3 diverse ones
        "lambda_mult": 0.5  # Balance between relevance and diversity
    }
)

query = "Tell me about RAG"
results = mmr_retriever.invoke(query)

print(f"🔍 MMR Results for: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    Topic: {doc.metadata['topic']}\n")

print("💡 Notice: Results are relevant but DIVERSE (covering different aspects of RAG)")

### 🎯 Exam Tip
> **When to use MMR vs Similarity:**
> - **Similarity search**: When you want the most relevant results, even if they're similar to each other
> - **MMR**: When you want **diverse** results that cover different aspects of the topic
> 
> `fetch_k` should be **larger than** `k` — it's the candidate pool from which diverse docs are selected.

## 4. Multi-Query Retriever

**Problem**: A single query may not capture all aspects of the user's intent.  
**Solution**: Use an LLM to generate **multiple variations** of the query, retrieve for each, and merge results.

```
Original: "How does RAG work?"
  ├─ Variation 1: "What are the components of a RAG pipeline?"
  ├─ Variation 2: "How does retrieval augmented generation process queries?"
  └─ Variation 3: "What is the architecture of RAG systems?"
     → Retrieve for each → Merge unique results
```

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

try:
    llm = ChatGroq(model="llama-3.1-8b-instant")

    multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        llm=llm
    )

    query = "How does RAG work?"
    results = multi_query_retriever.invoke(query)

    print(f"🔍 Multi-Query Results for: '{query}'")
    print(f"📄 Retrieved {len(results)} unique documents:\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] {doc.page_content[:80]}...")

except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Ensure GROQ_API_KEY is set in .env")

### 🎯 Exam Tip
> Multi-query retriever is useful when:
> - The query is **ambiguous** or **complex**
> - You want to **maximize recall** (find as many relevant docs as possible)
> - The user's intent could be interpreted in **multiple ways**
>
> Trade-off: **Slower** (makes LLM calls) but **higher recall**

## 5. Contextual Compression Retriever

**Problem**: Retrieved chunks may contain irrelevant parts.  
**Solution**: Use an LLM to **compress/extract** only the relevant portions from retrieved documents.

```
Retrieved chunk: "Python was created by Guido van Rossum. RAG is a technique 
that combines retrieval with generation. Python 3.12 was released in 2023."

Query: "What is RAG?"

Compressed: "RAG is a technique that combines retrieval with generation."
```

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

try:
    llm = ChatGroq(model="llama-3.1-8b-instant")
    
    # Create compressor
    compressor = LLMChainExtractor.from_llm(llm)
    
    # Create compressed retriever
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
    )
    
    query = "What retrieval methods balance relevance with diversity?"
    results = compression_retriever.invoke(query)
    
    print(f"🔍 Compressed Results for: '{query}'\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] {doc.page_content}")

except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Ensure GROQ_API_KEY is set in .env")

## 6. Ensemble Retriever (Hybrid Search)

Combines **multiple retrievers** and merges their results using **Reciprocal Rank Fusion (RRF)**.

### Common Pattern: Dense + Sparse
```
Dense Retriever (FAISS/Chroma) → Semantic similarity results
                                          ↓
                                   Ensemble/RRF Merge → Final ranked results
                                          ↑
Sparse Retriever (BM25)        → Keyword matching results
```

### Reciprocal Rank Fusion (RRF)
```
RRF_score(doc) = Σ 1 / (k + rank_in_retriever_i)
```
Documents that rank high in **multiple retrievers** get the highest final scores.

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Dense retriever (semantic)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse retriever (keyword-based BM25)
bm25_retriever = BM25Retriever.from_documents(docs, k=3)

# Ensemble (hybrid)
ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]  # Equal weight to both
)

query = "FAISS vector search"
results = ensemble_retriever.invoke(query)

print(f"🔍 Ensemble (Hybrid) Results for: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    Topic: {doc.metadata.get('topic', 'N/A')}\n")

### 🎯 Exam Tip
> Ensemble/Hybrid search is the **gold standard** for production RAG:  
> - Dense catches **semantic similarity** ("happy" ≈ "joyful")  
> - Sparse catches **exact keywords** ("FAISS" = "FAISS")  
> - Combined = catches both!  
>
> `weights=[0.5, 0.5]` gives equal importance to both retrievers.

## 7. ⭐ Retriever Comparison (EXAM CRITICAL!)

| Retriever | How it works | Pros | Cons | Use When |
|-----------|-------------|------|------|----------|
| **Similarity** | Nearest vectors by distance | Simple, fast | May return redundant results | Default choice |
| **MMR** | Relevance + diversity | Diverse results | Slightly slower | Need coverage of subtopics |
| **Multi-Query** | LLM generates query variations | Higher recall | Uses LLM (slower, costs) | Ambiguous/complex queries |
| **Compression** | LLM extracts relevant parts | More precise context | Uses LLM (slower, costs) | Long chunks with mixed content |
| **BM25** | Term frequency (sparse) | Great for keywords | Misses semantic meaning | Known exact terms |
| **Ensemble** | Combines multiple retrievers | Best of both worlds | More complex setup | Production systems |

### Search Type Parameter Values
```python
# For vectorstore.as_retriever()
search_type="similarity"           # Default cosine/L2 search
search_type="mmr"                  # Maximal Marginal Relevance
search_type="similarity_score_threshold"  # Only return docs above threshold
```

## 8. Metadata Filtering

In [ ]:
# Metadata filtering narrows retrieval to specific subsets
# This works best with ChromaDB and Qdrant

# Example: Only retrieve documents about RAG topic
try:
    from langchain_chroma import Chroma
    
    chroma_db = Chroma.from_documents(docs, embeddings, collection_name="filter_demo")
    
    # Filter by metadata
    filtered_retriever = chroma_db.as_retriever(
        search_kwargs={
            "k": 3,
            "filter": {"topic": "RAG"}  # Only search in RAG documents
        }
    )
    
    results = filtered_retriever.invoke("Tell me everything")
    print(f"🔍 Filtered results (topic=RAG only):\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] {doc.page_content}")
        print(f"    Topic: {doc.metadata['topic']}\n")

except ImportError:
    print("⚠️ langchain_chroma not installed.")
    print("\n💡 Metadata filtering syntax:")
    print('   search_kwargs={"filter": {"topic": "RAG"}}')

### 🎯 Exam Tip
> Metadata filtering is used when you need to **narrow search scope**:  
> - Only search in documents from a specific department  
> - Only search in documents from a specific date range  
> - Only search in documents of a specific category  
>
> This is different from **semantic search** — filtering happens BEFORE similarity computation.

## 🧠 Self-Assessment Quiz

---

**Q1.** Your RAG system keeps returning 3 very similar documents that all say the same thing. Which retriever would fix this?

<details>
<summary>Click for Answer</summary>

**MMR (Maximal Marginal Relevance)** — it balances relevance with diversity, ensuring retrieved documents cover different aspects. Set `search_type="mmr"` and use `lambda_mult` to control the relevance-diversity tradeoff.
</details>

---

**Q2.** What is the difference between `k` and `fetch_k` in MMR retrieval?

<details>
<summary>Click for Answer</summary>

- `fetch_k` = number of **candidate documents** fetched initially (larger pool)  
- `k` = number of **final documents** returned after diversity selection (smaller subset)  
- `fetch_k` should always be **larger than** `k` (e.g., fetch_k=20, k=5)
</details>

---

**Q3.** What does BM25 excel at that dense retrieval might miss?

<details>
<summary>Click for Answer</summary>

**Exact keyword matching**. Dense retrieval excels at semantic similarity (understanding meaning) but might not give high scores to documents containing the exact search terms. BM25 is based on term frequency and directly rewards documents with matching keywords.
</details>

---

**Q4.** Explain how Ensemble Retriever merges results from multiple retrievers.

<details>
<summary>Click for Answer</summary>

Ensemble Retriever uses **Reciprocal Rank Fusion (RRF)**. It collects results from each retriever, then scores documents based on their ranks across all retrievers:  
`RRF_score = Σ 1/(k + rank_i)`  
Documents that appear highly ranked in **multiple** retrievers get the highest combined scores.
</details>

---

**Q5.** When would you use a Contextual Compression Retriever?

<details>
<summary>Click for Answer</summary>

When your chunks contain **mixed content** — some relevant, some irrelevant. The compression retriever uses an LLM to **extract only the relevant portions**, reducing noise in the context. Trade-off: adds LLM latency and cost per retrieval.
</details>

---

## ✅ Module 6 Complete!

**Key Takeaways:**
1. Retriever quality is the #1 factor in RAG pipeline performance
2. **Similarity** = default, **MMR** = diverse results, **Ensemble** = best quality
3. Dense catches meaning, sparse (BM25) catches keywords, hybrid = both
4. Multi-query generates query variations for higher recall
5. Metadata filtering narrows search scope before similarity computation
6. `fetch_k > k` for MMR; `weights` control ensemble balance

**Next:** [Module 07 — RAG Chain End-to-End](./07_RAG_Chain_End2End.ipynb)